In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [15]:
results = pd.read_csv('Predicted scores.csv')

standings = {}
for _, row in results.iterrows():
    home, away, winner = row['Home team name'], row['away team name'], row['Predicted Results']

    standings.setdefault(home, {'Points':0, 'GD':0})
    standings.setdefault(away, {'Points':0, 'GD':0})

    if winner == home:
        standings[home]['Points'] += 3
    elif winner == away:
        standings[away]['Points'] += 3
    else:
        standings[home]['Points'] += 1
        standings[home]['Points'] += 1

standings_data = pd.DataFrame.from_dict(standings, orient='index').reset_index().rename(columns={'index':'Team'})
standings_data = standings_data.sort_values(by=['Points', 'GD'], ascending=False)

standings_data.to_csv('Fifa cup standings.csv', index=False)
print('ITS DONE')

ITS DONE


In [21]:
qualified_teams = standings_data.head(32)['Team'].tolist()
print(f'\n Qualified Teams to the KnockOut Stage: {qualified_teams}')


 Qualified Teams to the KnockOut Stage: ['Spain', 'Germany', 'Portugal', 'Mexico', 'Czech Republic', 'Canada', 'Brazil', 'USA', 'Ecuador', 'Belgium', 'France', 'Argentina', 'England', 'Switzerland', 'Scotland', 'Türkiye', 'Netherlands', 'Tunisia', 'New Zealand', 'Norway', 'Algeria', 'Colombia', 'Panama', 'Morocco', 'Japan', 'IR Iran', 'Uruguay', 'Austria', 'Croatia', 'South Africa', 'South Korea', 'Bosnia and Herzegovina']


# Modelling Knockout Stage upto final winner of this tournament

In [ ]:
data = pd.read_csv('Encoded data.csv')

home_model = joblib.load('home_score_model.pkl')
away_model = joblib.load('away_score_model.pkl')

knockout_rounds = ['Round of 32', 'Round of 16', 'Quarter-Final', 'Semi-Final', 'Third-Place Match', 'Final Match']
remaining_teams = qualified_teams.copy()

def predict_match(team_a, team_b, stage_name):
    
    match_data = data[
        ((data['Home Team Name'] == team_a) & (data['Away Team Name'] == team_b)) |
        ((data['Home Team Name'] == team_b) & (data['Away Team Name'] == team_a))
    ].copy()

    #incase direct data is not there
    if len(match_data) == 0:
        team_a_stats = data[data['Home Team Name'] == team_a].mean(numeric_only=True)
        team_b_stats = data[data['Home Team Name'] == team_b].mean(numeric_only=True)
        
        # Building feature row
        match_data = pd.DataFrame([{
            'Home Team Encoded': team_a_stats.get('Home Team Encoded', 0),
            'Away Team Encoded': team_b_stats.get('Away Team Encoded', 0),
            'rank_diff': team_a_stats.get('rank_diff', 0),
            'points_diff': team_a_stats.get('points_diff', 0),
            'points_ratio': team_a_stats.get('points_ratio', 1),
            'home_top5': 1 if team_a_stats.get('rank_home', 200) <=5 else 0,
            'home_top10': 1 if team_a_stats.get('rank_home', 200) <=10 else 0,
            'home_top20': 1 if team_a_stats.get('rank_home', 200) <=20 else 0,
            'away_top5': 1 if team_b_stats.get('rank_away', 200) <=5 else 0,
            'away_top10': 1 if team_b_stats.get('rank_away', 200) <=10 else 0,
            'away_top20': 1 if team_b_stats.get('rank_away', 200) <=20 else 0,
            'last_5score_home': team_a_stats.get('last_5score_home', 1.2),
            'last_5conceded_home': team_a_stats.get('last_5conceded_home', 1.2),
            'last_5score_away': team_b_stats.get('last_5score_away', 1.2),
            'last_5conceded_away': team_b_stats.get('last_5conceded_away', 1.2),
            'Home_attack_strength': team_a_stats.get('Home_attack_strength', 1),
            'Home_defence_strength': team_a_stats.get('Home_defence_strength', 1),
            'Away_attack_strength': team_b_stats.get('Away_attack_strength', 1),
            'Away_defence_strength': team_b_stats.get('Away_defence_strength', 1),
            'is_group': 0,
            'is_knockout': 1,
            'is_final': 1 if stage_name == 'Final' else 0,
            'h2h_count': 0,
            'h2h_avg_goals': 2.6,
            'h2h_home_win_rate': 0.48
        }])

    features = [
        'Home Team Encoded', 'Away Team Encoded', 'rank_diff', 'points_diff', 'points_ratio',
        'home_top5', 'home_top10', 'home_top20', 'away_top5', 'away_top10', 'away_top20',
        'last_5score_home', 'last_5conceded_home', 'last_5score_away', 'last_5conceded_away',
        'Home_attack_strength', 'Home_defence_strength', 'Away_attack_strength', 'Away_defence_strength',
        'is_group', 'is_knockout', 'is_final', 'h2h_count', 'h2h_avg_goals', 'h2h_home_win_rate'
    ]

    # Predict goals
    home_goals = round(max(0, home_model.predict(match_data[features])[0]))
    away_goals = round(max(0, away_model.predict(match_data[features])[0]))

    # Decide winner (penalties if draw)
    if home_goals > away_goals:
        winner = team_a
    elif away_goals > home_goals:
        winner = team_b
    else:
        # Penalty shootout: slight advantage to higher-ranked team
        rank_a = data[data['Home Team Name'] == team_a]['rank_home'].mean()
        rank_b = data[data['Home Team Name'] == team_b]['rank_home'].mean()
        winner = team_a if rank_a < rank_b else team_b

    return {
        'team_a': team_a,
        'team_b': team_b,
        'score': f"{home_goals} - {away_goals}",
        'winner': winner
    }

In [ ]:
print("🏆 STARTING 2026 WORLD CUP KNOCKOUT STAGES")
print("="*70)

for round_name in knockout_rounds:
    if len(remaining_teams) == 1:
        break  # already have champion
    
    print(f"\n👇 {round_name.upper()}")
    print("-"*50)
    
    next_round_teams = []
    
    # Pair teams together
    for i in range(0, len(remaining_teams), 2):
        if i + 1 >= len(remaining_teams):
            next_round_teams.append(remaining_teams[i])
            break
        
        team1 = remaining_teams[i]
        team2 = remaining_teams[i+1]
        
        # Predict match
        result = predict_match(team1, team2, round_name)
        
        print(f"⚽ {result['team_a']} vs {result['team_b']} → {result['score']} | Winner: {result['winner']}")
        next_round_teams.append(result['winner'])
    
    remaining_teams = next_round_teams
    print(f"\n✅ Teams advancing to next round: {', '.join(remaining_teams)}")

print("\n" + "="*70)
print(f"🏆 2026 WORLD CUP CHAMPION: {remaining_teams[0].upper()} 🎉🎉🎉")
print("="*70)

🏆 STARTING 2026 WORLD CUP KNOCKOUT STAGES

👇 ROUND OF 32
--------------------------------------------------
⚽ Spain vs Germany → 1 - 1 | Winner: Germany
⚽ Portugal vs Mexico → 2 - 1 | Winner: Portugal
⚽ Czech Republic vs Canada → 2 - 1 | Winner: Czech Republic
⚽ Brazil vs USA → 2 - 0 | Winner: Brazil
⚽ Ecuador vs Belgium → 1 - 1 | Winner: Belgium
⚽ France vs Argentina → 2 - 1 | Winner: France
⚽ England vs Switzerland → 2 - 1 | Winner: England
⚽ Scotland vs Türkiye → 1 - 1 | Winner: Türkiye
⚽ Netherlands vs Tunisia → 2 - 1 | Winner: Netherlands
⚽ New Zealand vs Norway → 1 - 1 | Winner: Norway
⚽ Algeria vs Colombia → 1 - 1 | Winner: Colombia
⚽ Panama vs Morocco → 1 - 1 | Winner: Morocco
⚽ Japan vs IR Iran → 1 - 1 | Winner: IR Iran
⚽ Uruguay vs Austria → 2 - 0 | Winner: Uruguay
⚽ Croatia vs South Africa → 2 - 0 | Winner: Croatia
⚽ South Korea vs Bosnia and Herzegovina → 1 - 1 | Winner: Bosnia and Herzegovina

✅ Teams advancing to next round: Germany, Portugal, Czech Republic, Brazil, Belg

--- 